# V10.2: SSL Pretrain + Text Fine-tune

**The cross-domain approach.** Same BraTS2021 data, different task.

| Stage | Task | Data | Labels | Text |
|-------|------|------|--------|------|
| **SSL Pretrain** | Masked reconstruction | BraTS2021 (1251) | **NO** | NO |
| **Fine-tune** | Segmentation | BraTS2020 (295) | YES | **YES** |

Backbone learns brain anatomy but NOT tumor segmentation.
Text fills the knowledge gap: 'where are the tumors?'

This mirrors TextBraTS's SwinUNETR approach:
- SwinUNETR: ImageNet (general vision) → text guides tumor segmentation
- V10.2: BraTS2021 SSL (brain anatomy) → text guides tumor segmentation


In [2]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi 2>/dev/null || echo 'No GPU'
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0: break
        if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else: raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)

# BraTS2021
BRATS2021_LOCAL = '/content/BraTS2021'
BRATS2021_ZIP = os.path.join(DRIVE_BASE, 'BraTS2021_archive.zip')
if os.path.exists(os.path.join(BRATS2021_LOCAL, 'train')):
    print(f'BraTS2021 already extracted')
else:
    print('Extracting BraTS2021...')
    os.makedirs('/content/brats2021_tmp', exist_ok=True)
    !unzip -q {BRATS2021_ZIP} -d /content/brats2021_tmp/
    !tar xf /content/brats2021_tmp/BraTS2021_Training_Data.tar -C /content/brats2021_tmp/
    os.makedirs(BRATS2021_LOCAL, exist_ok=True)
    !mv /content/brats2021_tmp/BraTS2021_* {BRATS2021_LOCAL}/ 2>/dev/null; true
    !python scripts/prepare_brats.py --input {BRATS2021_LOCAL} --output {BRATS2021_LOCAL}
    !rm -rf /content/brats2021_tmp

def sync_and_tag(tag):
    lc = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(lc): return
    for f in glob.glob(os.path.join(lc, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    b = os.path.join(lc, 'best.pth')
    if os.path.exists(b): shutil.copy2(b, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
    l = os.path.join(lc, 'last.pth')
    if os.path.exists(l): shutil.copy2(l, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')


Mounted at /content/drive
Fri Apr 10 05:46:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## Stage 1: SSL Pretrain (Masked Reconstruction, No Labels)

200 epochs on BraTS2021 images. 50% patch masking, L1 reconstruction loss.


In [2]:
import os, shutil, torch
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Check if SSL pretrain is complete (epoch >= 199)
def get_ssl_epoch(path):
    if not os.path.exists(path):
        return -1
    try:
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        return ckpt.get('epoch', -1)
    except Exception as e:
        print(f'Could not read {path}: {e}')
        return -1

drive_ssl_last = os.path.join(DRIVE_CKPT, 'last_ssl.pth')
drive_ssl_best = os.path.join(DRIVE_CKPT, 'best_ssl.pth')

last_epoch = get_ssl_epoch(drive_ssl_last)
print(f'SSL last checkpoint on Drive: epoch={last_epoch}')

if last_epoch >= 199:
    print(f'SSL pretrain already complete (epoch {last_epoch}). Skip to Stage 2.')
else:
    # Prepare local checkpoint for resume
    os.makedirs('checkpoints', exist_ok=True)
    resume_arg = ''
    if last_epoch >= 0:
        shutil.copy2(drive_ssl_last, 'checkpoints/last_ssl.pth')
        if os.path.exists(drive_ssl_best):
            shutil.copy2(drive_ssl_best, 'checkpoints/best_ssl.pth')
        resume_arg = '--resume checkpoints/last_ssl.pth'
        print(f'Resuming SSL pretrain from epoch {last_epoch}')
    else:
        print('Starting SSL pretrain fresh')

    !python -u scripts/pretrain_ssl.py \
        --data-dir /content/BraTS2021 \
        --epochs 200 \
        --batch-size 4 \
        --lr 1e-4 \
        --mask-ratio 0.5 \
        --embed-dim 48 \
        {resume_arg}

    # Sync SSL checkpoints to Drive
    for name in ['best_ssl.pth', 'last_ssl.pth']:
        src = os.path.join('checkpoints', name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_CKPT, name))
    print('SSL pretrain complete!')


SSL last checkpoint on Drive: epoch=99
Resuming SSL pretrain from epoch 99
/content/TextMamba3D/models/__init__.py:1: UserWarning: Real Mamba3 not available, falling back to Mamba2.
  from .mamba_block import (
Device: cuda
Training samples: 875
/content/TextMamba3D/models/mamba_block.py:357: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.dhw_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:358: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.hwd_fwd = _create_ssm(**ssm_kw)
/content/TextMamba3D/models/mamba_block.py:359: UserWarning: Mamba3 backward crashes for d_model=48 < 96. Falling back to Mamba-2 for this layer.
  self.wdh_fwd = _create_ssm(**ssm_kw)
Resumed from epoch 100
SSL Pretraining: 200 epochs, mask_ratio=0.5
Encoder params: 9,394,982
Epoch 100: 100% 218/218 [05:42<00:00,  1.57s/it, loss=0.1527, lr=5.41e-05] 
Epoch 100: train_loss=0.1745, 

In [3]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
lc = os.path.join(REPO_DIR, 'checkpoints')
fs = glob.glob(os.path.join(lc, '*.pth'))
if fs:
    for f in sorted(fs):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    subprocess.run(['sync'], check=True)
    print(f'{len(fs)} files synced')
else: print('No checkpoints')


No checkpoints


## Stage 2: Fine-tune with Text on BraTS2020

Load SSL-pretrained encoder weights into TextMamba3D, then fine-tune
with SeqCA text fusion on BraTS2020+TextBraTS.


In [4]:
import os, glob, torch, shutil
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)

V102_BEST = os.path.join(DRIVE_CKPT, 'best_V10.2.pth')
V102_LAST = os.path.join(DRIVE_CKPT, 'last_V10.2.pth')
SSL_CKPT = os.path.join(DRIVE_CKPT, 'best_ssl.pth')

# Check if SSL is newer than existing V10.2 (means SSL was re-trained)
need_retrain = False
if os.path.exists(V102_BEST) and os.path.exists(SSL_CKPT):
    ssl_time = os.path.getmtime(SSL_CKPT)
    v102_time = os.path.getmtime(V102_BEST)
    if ssl_time > v102_time:
        print('SSL checkpoint is NEWER than V10.2 — need to re-run Stage 2')
        os.remove(V102_BEST)
        if os.path.exists(V102_LAST):
            os.remove(V102_LAST)
        need_retrain = True
    else:
        print(f'V10.2 already complete: {V102_BEST}')
        print('Skip to evaluation')
elif os.path.exists(V102_BEST):
    print(f'V10.2 already complete: {V102_BEST}')
    print('Skip to evaluation')
else:
    need_retrain = True

if need_retrain:
    if os.path.exists(V102_LAST):
        # Resume interrupted Stage 2
        shutil.copy2(V102_LAST, os.path.join(ckpt_dir, 'last.pth'))
        print(f'Resuming V10.2 Stage 2 from: {V102_LAST}')
        !python -u train.py \
            --config configs/autoresearch/V10.2_ssl_pretrain.yaml \
            --resume checkpoints/last.pth \
            --no-text-ratio 0.15 \
            --grad-accum 2
        sync_and_tag('V10.2')
        print('V10.2 Stage 2 complete!')
    else:
        # First run: create SSL-initialized checkpoint
        assert os.path.exists(SSL_CKPT), f'SSL checkpoint not found: {SSL_CKPT}'
        ssl_state = torch.load(SSL_CKPT, map_location='cpu', weights_only=False)
        encoder_state = ssl_state['encoder']

        from models.textmamba3d import TextMamba3D
        model = TextMamba3D(
            img_size=(128,128,128), embed_dim=48, depths=[2,2,2,2],
            text_embed_dim=256, use_mamba3=True, headdim=48,
            fusion_type='seqca',
        )
        full_state = model.state_dict()
        loaded = 0
        for k, v in encoder_state.items():
            full_key = f'img_encoder.{k}'
            if full_key in full_state and full_state[full_key].shape == v.shape:
                full_state[full_key] = v
                loaded += 1
        print(f'Loaded {loaded} encoder params from SSL checkpoint')

        for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
            os.remove(f)
        ssl_resume = os.path.join(ckpt_dir, 'ssl_init.pth')
        torch.save({'model': full_state, 'epoch': -1, 'best_dice': 0, 'best_dice_no_text': 0}, ssl_resume)

        print('Stage 2: Fine-tune with text (from 200-epoch SSL)')
        !python -u train.py \
            --config configs/autoresearch/V10.2_ssl_pretrain.yaml \
            --resume "{ssl_resume}" \
            --reset-optimizer \
            --no-text-ratio 0.15 \
            --grad-accum 2
        sync_and_tag('V10.2')
        print('V10.2 Stage 2 complete!')


## Evaluation


In [ ]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V10.2.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V10.2_ssl_pretrain.yaml'
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG, '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Comparison:')
print('  V5.0  (scratch, SeqCA):     Mean=0.8479, delta=+0.55%')
print('  V8.0  (sup pretrain):       Mean=0.8753, delta=0.00%')
print('  V10.2 (SSL pretrain+text):  Mean=?, delta=?')
print('  TextBraTS (SwinUNETR+IN):   Mean=0.853, delta=+1.5%')


Evaluating: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V10.2.pth

text+TTA
Loaded checkpoint: epoch=84, best_dice=0.8230303820268609
TextBraTS test: 95 samples

Evaluating 95 cases (test split)
Sliding window: patch=(128, 128, 128), overlap=0.5, text=True
TTA: 8-fold flip ensemble ENABLED

  BraTS20_Training_328: Dice=0.6287 (ET=0.2605, TC=0.7227, WT=0.9030) HD95_ET=51.49
  BraTS20_Training_028: Dice=0.7105 (ET=0.5434, TC=0.8302, WT=0.7580) HD95_ET=3.16
  BraTS20_Training_289: Dice=0.5310 (ET=0.0000, TC=0.6794, WT=0.9138) HD95_ET=nan
  BraTS20_Training_231: Dice=0.9079 (ET=0.8818, TC=0.9100, WT=0.9320) HD95_ET=1.00
  BraTS20_Training_261: Dice=0.6324 (ET=0.2293, TC=0.7339, WT=0.9340) HD95_ET=29.64
  BraTS20_Training_163: Dice=0.8679 (ET=0.7690, TC=0.8910, WT=0.9438) HD95_ET=2.00
  BraTS20_Training_345: Dice=0.5796 (ET=0.7759, TC=0.2447, WT=0.7184) HD95_ET=6.08
  BraTS20_Training_139: Dice=0.8315 (ET=0.7975, TC=0.8463, WT=0.8506) HD95_ET=1.00
  BraTS20_Training_063: Dice=0.7186